In [1]:
# =============================================================================
# IPL DATA ANALYSIS - COMPLETE JUPYTER NOTEBOOK CODE
# Covers: EDA | NumPy | Pandas | MySQL | Power BI Export
# Dataset: ipl-matches.csv (2008-2022, 950 matches)
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# CELL 1: INSTALL & IMPORT LIBRARIES
# ─────────────────────────────────────────────────────────────────────────────
# !pip install pandas numpy matplotlib seaborn plotly sqlalchemy mysql-connector-python openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import os

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.facecolor'] = '#1a1a2e'
plt.rcParams['figure.facecolor'] = '#0f0f23'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['axes.titlecolor'] = 'white'

COLORS = ['#4fc3f7', '#f9a825', '#ab47bc', '#ef5350', '#26a69a',
          '#ff7043', '#66bb6a', '#ec407a', '#29b6f6', '#d4e157']
IPL_BLUE  = '#1a1a2e'
IPL_GOLD  = '#f9a825'
IPL_GREEN = '#4caf50'

print("✅ Libraries imported successfully!")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 2: LOAD & PREVIEW DATA
# ─────────────────────────────────────────────────────────────────────────────
df = pd.read_csv('ipl-matches.csv')          # ← adjust path if needed

print("=" * 55)
print("       IPL DATASET OVERVIEW")
print("=" * 55)
print(f"  Rows          : {df.shape[0]:,}")
print(f"  Columns       : {df.shape[1]}")
print(f"  Seasons       : {df['Season'].nunique()}  ({sorted(df['Season'].unique())[0]} – {sorted(df['Season'].unique())[-1]})")
print(f"  Teams         : {pd.concat([df['Team1'], df['Team2']]).nunique()}")
print(f"  Venues        : {df['Venue'].nunique()}")
print(f"  Cities        : {df['City'].nunique()}")
print("=" * 55)
df.head(3)


# ─────────────────────────────────────────────────────────────────────────────
# CELL 3: DATA TYPES & COLUMN INFO
# ─────────────────────────────────────────────────────────────────────────────
print("\n📋 COLUMN INFO")
df.info()
print("\n📊 NUMERIC SUMMARY")
df.describe().T


# ─────────────────────────────────────────────────────────────────────────────
# CELL 4: DATA CLEANING & FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
# -- 4a. Parse dates
df['Date'] = pd.to_datetime(df['Date'])
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.day_name()

# -- 4b. Normalise season to a 4-digit year
def season_year(s):
    s = str(s)
    if '/' in s:                     # e.g. "2007/08" or "2020/21"
        return int(s.split('/')[0])
    return int(s)

df['SeasonYear'] = df['Season'].apply(season_year)

# -- 4c. Did the toss winner also win the match?
df['TossWinMatchWin'] = (df['TossWinner'] == df['WinningTeam']).astype(int)

# -- 4d. Clean margins: coerce non-numeric to NaN
df['Margin'] = pd.to_numeric(df['Margin'], errors='coerce')

# -- 4e. Fill missing City using Venue
venue_city_map = (
    df.dropna(subset=['City'])
      .drop_duplicates(subset=['Venue'])
      .set_index('Venue')['City']
      .to_dict()
)
df['City'] = df.apply(
    lambda r: venue_city_map.get(r['Venue'], r['City']) if pd.isna(r['City']) else r['City'],
    axis=1
)

# -- 4f. Strip venue to short name (remove city suffix after comma)
df['VenueShort'] = df['Venue'].str.split(',').str[0].str.strip()

# -- 4g. Flag Super Over matches
df['IsSuperOver'] = df['SuperOver'].str.strip().str.upper().eq('Y')

print("✅ Cleaning done. New columns:", [c for c in df.columns if c not in
      ['ID','City','Date','Season','MatchNumber','Team1','Team2','Venue',
       'TossWinner','TossDecision','SuperOver','WinningTeam','WonBy','Margin',
       'method','Player_of_Match','Team1Players','Team2Players','Umpire1','Umpire2']])
df[['Date','SeasonYear','VenueShort','TossWinMatchWin','IsSuperOver']].head()


# ─────────────────────────────────────────────────────────────────────────────
# CELL 5: MISSING VALUE ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
miss = df.isnull().sum().sort_values(ascending=False)
miss = miss[miss > 0]
bars = ax.barh(miss.index, miss.values, color=IPL_GOLD, edgecolor='#fff', linewidth=0.5)
for bar, val in zip(bars, miss.values):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'{val} ({val/len(df)*100:.1f}%)', va='center', color='white', fontsize=10)
ax.set_title('Missing Values by Column', fontsize=15, color=IPL_GOLD, pad=12)
ax.set_xlabel('Count')
ax.set_facecolor(IPL_BLUE)
fig.patch.set_facecolor(IPL_BLUE)
plt.tight_layout()
plt.savefig('01_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ Remaining nulls in key columns:\n{df[['WinningTeam','Margin','City']].isnull().sum()}")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 6: NUMPY STATISTICAL ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
runs_arr    = df[df['WonBy'] == 'Runs']['Margin'].dropna().values
wickets_arr = df[df['WonBy'] == 'Wickets']['Margin'].dropna().values

print("=" * 45)
print("  NUMPY STATS — WIN BY RUNS")
print("=" * 45)
print(f"  Count   : {len(runs_arr)}")
print(f"  Mean    : {np.mean(runs_arr):.2f}")
print(f"  Median  : {np.median(runs_arr):.2f}")
print(f"  Std Dev : {np.std(runs_arr):.2f}")
print(f"  Min     : {np.min(runs_arr)}")
print(f"  Max     : {np.max(runs_arr)}")
print(f"  25th %  : {np.percentile(runs_arr, 25):.1f}")
print(f"  75th %  : {np.percentile(runs_arr, 75):.1f}")

print("\n" + "=" * 45)
print("  NUMPY STATS — WIN BY WICKETS")
print("=" * 45)
print(f"  Count   : {len(wickets_arr)}")
print(f"  Mean    : {np.mean(wickets_arr):.2f}")
print(f"  Median  : {np.median(wickets_arr):.2f}")
print(f"  Std Dev : {np.std(wickets_arr):.2f}")
print(f"  Min     : {np.min(wickets_arr)}")
print(f"  Max     : {np.max(wickets_arr)}")

# Correlation matrix using pandas (handles NaN cleanly)
corr_df = df[['SeasonYear','Margin','TossWinMatchWin']].dropna()
corr_matrix = np.corrcoef(corr_df.values.T)
print("\n📐 Correlation matrix (Season / Margin / TossWinMatchWin):")
print(pd.DataFrame(corr_matrix, index=corr_df.columns, columns=corr_df.columns).round(3))


# ─────────────────────────────────────────────────────────────────────────────
# CELL 7: MATCHES PER SEASON
# ─────────────────────────────────────────────────────────────────────────────
matches_per_season = df.groupby('SeasonYear').size().reset_index(name='Matches')

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(matches_per_season['SeasonYear'].astype(str),
              matches_per_season['Matches'], color='#4fc3f7', edgecolor='white', linewidth=0.6)
for bar, val in zip(bars, matches_per_season['Matches']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', va='bottom', color='white', fontsize=9, fontweight='bold')
ax.set_title('Matches Per Season (2008–2022)', fontsize=15, color=IPL_GOLD, pad=12)
ax.set_xlabel('Season')
ax.set_ylabel('Number of Matches')
ax.set_facecolor(IPL_BLUE)
fig.patch.set_facecolor(IPL_BLUE)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('02_matches_per_season.png', dpi=150, bbox_inches='tight')
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# CELL 8: TEAM WIN ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
team_wins = (df.dropna(subset=['WinningTeam'])
               .groupby('WinningTeam')
               .size()
               .sort_values(ascending=False)
               .reset_index(name='Wins'))

# Total matches played
team1 = df.groupby('Team1').size()
team2 = df.groupby('Team2').size()
total_played = (team1.add(team2, fill_value=0)).reset_index()
total_played.columns = ['Team', 'Played']
team_wins = team_wins.rename(columns={'WinningTeam': 'Team'})
team_wins = team_wins.merge(total_played, on='Team')
team_wins['WinRate'] = (team_wins['Wins'] / team_wins['Played'] * 100).round(1)
team_wins['Losses']  = team_wins['Played'] - team_wins['Wins']

print(team_wins.to_string(index=False))

# Chart — Top 10 teams
top10 = team_wins.head(10)
fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.barh(top10['Team'][::-1], top10['Wins'][::-1],
               color=COLORS[:10], edgecolor='white', linewidth=0.5)
for bar, val, wr in zip(bars, top10['Wins'][::-1], top10['WinRate'][::-1]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{val}  ({wr}%)', va='center', color='white', fontsize=9)
ax.set_title('Top 10 Teams by Total Wins', fontsize=15, color=IPL_GOLD, pad=12)
ax.set_xlabel('Number of Wins')
ax.set_facecolor(IPL_BLUE)
fig.patch.set_facecolor(IPL_BLUE)
plt.tight_layout()
plt.savefig('03_team_wins.png', dpi=150, bbox_inches='tight')
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# CELL 9: TOSS ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
# 9a. Toss decision distribution
toss_dec = df['TossDecision'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie
axes[0].pie(toss_dec.values, labels=[f'{l.title()}\n{v/toss_dec.sum()*100:.1f}%'
            for l, v in zip(toss_dec.index, toss_dec.values)],
            colors=['#4fc3f7', '#f9a825'], startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Toss Decision Distribution', fontsize=13, color=IPL_GOLD)
axes[0].set_facecolor(IPL_BLUE)

# Toss winner = match winner?
toss_effect = df.groupby('TossDecision')['TossWinMatchWin'].mean() * 100
axes[1].bar(toss_effect.index, toss_effect.values, color=['#4fc3f7','#f9a825'],
            edgecolor='white', linewidth=0.7)
for i, (idx, val) in enumerate(toss_effect.items()):
    axes[1].text(i, val + 0.5, f'{val:.1f}%', ha='center', color='white', fontsize=12, fontweight='bold')
axes[1].set_title('% Matches Won by Toss Winner\n(by Toss Decision)', fontsize=12, color=IPL_GOLD)
axes[1].set_xlabel('Toss Decision')
axes[1].set_ylabel('Win % by Toss Winner')
axes[1].set_ylim(0, 70)
axes[1].set_facecolor(IPL_BLUE)

for ax in axes:
    ax.tick_params(colors='white')
fig.patch.set_facecolor(IPL_BLUE)
plt.tight_layout()
plt.savefig('04_toss_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

overall_toss_win_pct = df['TossWinMatchWin'].mean() * 100
print(f"\n📌 Overall: Toss winner won the match {overall_toss_win_pct:.1f}% of the time")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 10: WIN TYPE ANALYSIS (RUNS vs WICKETS)
# ─────────────────────────────────────────────────────────────────────────────
won_by_counts = df['WonBy'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Donut
wedges, texts, autotexts = axes[0].pie(
    won_by_counts.values,
    labels=won_by_counts.index,
    autopct='%1.1f%%',
    colors=COLORS,
    startangle=90,
    pctdistance=0.75,
    wedgeprops={'edgecolor':'white','linewidth':1.5,'width':0.55}
)
for t in autotexts: t.set_color('white'); t.set_fontsize(11)
axes[0].set_title('Win Type Distribution', fontsize=13, color=IPL_GOLD)

# Margin distribution
runs_data    = df[df['WonBy'] == 'Runs']['Margin'].dropna()
wickets_data = df[df['WonBy'] == 'Wickets']['Margin'].dropna()
axes[1].hist(runs_data, bins=25, alpha=0.8, color='#4fc3f7', label='Runs', edgecolor='white', linewidth=0.4)
axes[1].hist(wickets_data, bins=10, alpha=0.8, color='#f9a825', label='Wickets', edgecolor='white', linewidth=0.4)
axes[1].set_title('Margin Distribution', fontsize=13, color=IPL_GOLD)
axes[1].set_xlabel('Margin')
axes[1].set_ylabel('Frequency')
axes[1].legend(facecolor='#222', labelcolor='white')
axes[1].set_facecolor(IPL_BLUE)

for ax in axes:
    ax.set_facecolor(IPL_BLUE)
    ax.tick_params(colors='white')
fig.patch.set_facecolor(IPL_BLUE)
plt.tight_layout()
plt.savefig('05_win_type.png', dpi=150, bbox_inches='tight')
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# CELL 11: VENUE ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
venue_matches = df.groupby('VenueShort').size().sort_values(ascending=False).head(12)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(venue_matches.index[::-1], venue_matches.values[::-1],
               color=plt.cm.plasma(np.linspace(0.3, 0.9, len(venue_matches))), edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, venue_matches.values[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', color='white', fontsize=10)
ax.set_title('Top 12 Venues by Matches Hosted', fontsize=15, color=IPL_GOLD, pad=12)
ax.set_xlabel('Number of Matches')
ax.set_facecolor(IPL_BLUE)
fig.patch.set_facecolor(IPL_BLUE)
plt.tight_layout()
plt.savefig('06_venue_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# CELL 12: PLAYER OF THE MATCH ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
potm = df['Player_of_Match'].value_counts().head(15).reset_index()
potm.columns = ['Player', 'Awards']

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.barh(potm['Player'][::-1], potm['Awards'][::-1],
               color=IPL_GOLD, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, potm['Awards'][::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', color='white', fontsize=10, fontweight='bold')
ax.set_title('Top 15 – Player of the Match Awards', fontsize=15, color=IPL_GOLD, pad=12)
ax.set_xlabel('No. of Awards')
ax.set_facecolor(IPL_BLUE)
fig.patch.set_facecolor(IPL_BLUE)
plt.tight_layout()
plt.savefig('07_potm.png', dpi=150, bbox_inches='tight')
plt.show()

print(potm.to_string(index=False))


# ─────────────────────────────────────────────────────────────────────────────
# CELL 13: TEAM PERFORMANCE OVER THE YEARS (LINE CHART)
# ─────────────────────────────────────────────────────────────────────────────
TOP_TEAMS = ['Mumbai Indians', 'Chennai Super Kings', 'Kolkata Knight Riders',
             'Royal Challengers Bangalore', 'Sunrisers Hyderabad', 'Rajasthan Royals']

wins_season = (df[df['WinningTeam'].isin(TOP_TEAMS)]
               .groupby(['SeasonYear', 'WinningTeam'])
               .size()
               .reset_index(name='Wins'))

fig, ax = plt.subplots(figsize=(14, 6))
palette = dict(zip(TOP_TEAMS, COLORS))
for team in TOP_TEAMS:
    data = wins_season[wins_season['WinningTeam'] == team]
    ax.plot(data['SeasonYear'], data['Wins'], marker='o', linewidth=2,
            label=team, color=palette[team])
    ax.annotate(team.split()[-1], xy=(data['SeasonYear'].iloc[-1], data['Wins'].iloc[-1]),
                color=palette[team], fontsize=8, va='center')

ax.set_title('Top Teams – Wins Per Season', fontsize=15, color=IPL_GOLD, pad=12)
ax.set_xlabel('Season')
ax.set_ylabel('Wins')
ax.legend(facecolor='#222', labelcolor='white', fontsize=8, loc='upper left')
ax.set_facecolor(IPL_BLUE)
fig.patch.set_facecolor(IPL_BLUE)
plt.tight_layout()
plt.savefig('08_team_over_years.png', dpi=150, bbox_inches='tight')
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# CELL 14: AVERAGE MARGIN BY SEASON
# ─────────────────────────────────────────────────────────────────────────────
avg_margin = df.groupby('SeasonYear')['Margin'].mean().reset_index()
avg_margin.columns = ['Season', 'AvgMargin']

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(avg_margin['Season'], avg_margin['AvgMargin'], alpha=0.3, color='#26a69a')
ax.plot(avg_margin['Season'], avg_margin['AvgMargin'], color='#26a69a',
        linewidth=2.5, marker='o')
for _, row in avg_margin.iterrows():
    ax.text(row['Season'], row['AvgMargin'] + 0.3, f"{row['AvgMargin']:.1f}",
            ha='center', color='white', fontsize=9)
ax.set_title('Average Match Margin Per Season', fontsize=15, color=IPL_GOLD, pad=12)
ax.set_xlabel('Season')
ax.set_ylabel('Avg Margin (Runs/Wickets)')
ax.set_facecolor(IPL_BLUE)
fig.patch.set_facecolor(IPL_BLUE)
plt.tight_layout()
plt.savefig('09_avg_margin.png', dpi=150, bbox_inches='tight')
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# CELL 15: SUPER OVER MATCHES
# ─────────────────────────────────────────────────────────────────────────────
so_matches = df[df['IsSuperOver'] == True].copy()
print(f"🔥 Total Super Over matches: {len(so_matches)}")
print("\nSuper Over matches by season:")
print(so_matches.groupby('SeasonYear').size().reset_index(name='SuperOvers').to_string(index=False))
print("\nTeams in Super Overs (most involved):")
so_teams = pd.concat([so_matches['Team1'], so_matches['Team2']]).value_counts().head(8)
print(so_teams)


# ─────────────────────────────────────────────────────────────────────────────
# CELL 16: MATCH DAY DISTRIBUTION
# ─────────────────────────────────────────────────────────────────────────────
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_dist  = df['DayOfWeek'].value_counts().reindex(day_order, fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(day_dist.index, day_dist.values, color=COLORS[:7], edgecolor='white', linewidth=0.5)
for i, val in enumerate(day_dist.values):
    ax.text(i, val + 1, str(val), ha='center', color='white', fontsize=10, fontweight='bold')
ax.set_title('Matches by Day of Week', fontsize=15, color=IPL_GOLD, pad=12)
ax.set_ylabel('Number of Matches')
ax.set_facecolor(IPL_BLUE)
fig.patch.set_facecolor(IPL_BLUE)
plt.tight_layout()
plt.savefig('10_day_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# CELL 17: HEAD-TO-HEAD HEATMAP (TOP 8 TEAMS)
# ─────────────────────────────────────────────────────────────────────────────
top8 = team_wins.head(8)['Team'].tolist()
h2h  = pd.DataFrame(0, index=top8, columns=top8)

for _, row in df.dropna(subset=['WinningTeam']).iterrows():
    w = row['WinningTeam']
    l = row['Team1'] if row['Team2'] == w else row['Team2']
    if w in top8 and l in top8:
        h2h.loc[w, l] += 1

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(h2h, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5,
            linecolor='#333', ax=ax, cbar_kws={'label': 'Wins'})
ax.set_title('Head-to-Head Win Matrix (Top 8 Teams)\n[Row = Winner, Col = Loser]',
             fontsize=13, color='black', pad=12)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig('11_h2h_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# CELL 18: CITY-LEVEL ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
city_matches = df.groupby('City').size().sort_values(ascending=False).head(15).reset_index()
city_matches.columns = ['City', 'Matches']

fig = px.bar(city_matches, x='City', y='Matches', color='Matches',
             color_continuous_scale='Blues', title='Top 15 Cities by Matches Hosted',
             template='plotly_dark', text='Matches')
fig.update_traces(textposition='outside')
fig.update_layout(title_font_color=IPL_GOLD, showlegend=False)
fig.write_html('12_city_matches.html')
fig.show()
print("✅ Interactive chart saved: 12_city_matches.html")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 19: INTERACTIVE DASHBOARD (PLOTLY)
# ─────────────────────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=(
        'Matches Per Season',
        'Top 8 Team Wins',
        'Win By Runs vs Wickets',
        'Toss Decision Split',
        'Top 10 Venues',
        'Avg Margin by Season',
        'Player of Match (Top 10)',
        'Team Wins vs Win Rate',
        'Matches by Month'
    ),
    specs=[[{},         {},         {'type':'pie'}],
           [{'type':'pie'}, {},     {}             ],
           [{},         {'type':'scatter'}, {}     ]]
)

# 1. Matches per season
mps = df.groupby('SeasonYear').size().reset_index(name='Matches')
fig.add_trace(go.Bar(x=mps['SeasonYear'].astype(str), y=mps['Matches'],
                     marker_color='#4fc3f7', name='Matches/Season'), row=1, col=1)

# 2. Top 8 teams
t8 = team_wins.head(8)
fig.add_trace(go.Bar(x=t8['Team'], y=t8['Wins'],
                     marker_color=COLORS[:8], name='Team Wins'), row=1, col=2)

# 3. Win type pie
fig.add_trace(go.Pie(labels=won_by_counts.index, values=won_by_counts.values,
                     name='WonBy', hole=0.5), row=1, col=3)

# 4. Toss decision pie
fig.add_trace(go.Pie(labels=toss_dec.index, values=toss_dec.values,
                     name='TossDec', hole=0.5), row=2, col=1)

# 5. Top 10 venues
v10 = venue_matches.head(10)
fig.add_trace(go.Bar(x=v10.values, y=v10.index, orientation='h',
                     marker_color='#ab47bc', name='Venues'), row=2, col=2)

# 6. Average margin
fig.add_trace(go.Scatter(x=avg_margin['Season'], y=avg_margin['AvgMargin'],
                          mode='lines+markers', line=dict(color='#26a69a', width=2),
                          name='Avg Margin'), row=2, col=3)

# 7. POTM top 10
p10 = potm.head(10)
fig.add_trace(go.Bar(x=p10['Awards'], y=p10['Player'], orientation='h',
                     marker_color=IPL_GOLD, name='POTM'), row=3, col=1)

# 8. Wins vs Win Rate scatter
fig.add_trace(go.Scatter(x=team_wins['Wins'], y=team_wins['WinRate'],
                          mode='markers+text', text=team_wins['Team'].str.split().str[-1],
                          textposition='top center', marker=dict(size=10, color=COLORS),
                          name='Wins vs Rate'), row=3, col=2)

# 9. Matches by month
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',7:'Jul',
             8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
mth = df['Month'].value_counts().sort_index()
fig.add_trace(go.Bar(x=[month_map[m] for m in mth.index], y=mth.values,
                     marker_color='#ef5350', name='By Month'), row=3, col=3)

fig.update_layout(
    height=1100, width=1300,
    title_text='🏏 IPL Complete Analytics Dashboard',
    title_font=dict(size=20, color=IPL_GOLD),
    paper_bgcolor='#0f0f23',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white'),
    showlegend=False
)
fig.write_html('13_ipl_dashboard.html')
fig.show()
print("✅ Interactive dashboard saved: 13_ipl_dashboard.html")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 20: PANDAS ADVANCED AGGREGATIONS
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 55)
print("  ADVANCED PANDAS AGGREGATIONS")
print("=" * 55)

# Pivot: wins per team per season
pivot_wins = (df.dropna(subset=['WinningTeam'])
               .groupby(['SeasonYear','WinningTeam'])
               .size()
               .unstack(fill_value=0))
print("\n📊 Pivot – Wins per Team per Season (Top 6 teams):")
cols_to_show = team_wins.head(6)['Team'].tolist()
print(pivot_wins[[c for c in cols_to_show if c in pivot_wins.columns]].to_string())

# Toss-to-win conversion by team
toss_win_team = (df.dropna(subset=['WinningTeam'])
                   .groupby('TossWinner')
                   .agg(TossWins=('TossWinMatchWin','sum'),
                        TotalToss=('TossWinMatchWin','count'))
                   .assign(ConvRate=lambda x: (x['TossWins']/x['TotalToss']*100).round(1))
                   .sort_values('ConvRate', ascending=False)
                   .head(10))
print("\n🎯 Toss-to-Win Conversion by Team (Top 10):")
print(toss_win_team.to_string())


# ─────────────────────────────────────────────────────────────────────────────
# CELL 21: EXPORT DATA FOR POWER BI
# ─────────────────────────────────────────────────────────────────────────────
# Build all summary tables and export to Excel (Power BI can connect directly)

# Table 1: Cleaned master
df_clean = df.drop(columns=['Team1Players','Team2Players'], errors='ignore')

# Table 2: Team summary
team_summary = team_wins.copy()

# Table 3: Season summary
season_summary = df.groupby('SeasonYear').agg(
    Matches       = ('ID', 'count'),
    AvgMargin     = ('Margin', 'mean'),
    TossWinPct    = ('TossWinMatchWin', 'mean'),
    SuperOvers    = ('IsSuperOver', 'sum')
).round(2).reset_index()

# Table 4: Venue summary
venue_summary = (df.groupby(['VenueShort','City'])
                   .size()
                   .reset_index(name='Matches')
                   .sort_values('Matches', ascending=False))

# Table 5: POTM leaderboard
potm_full = df['Player_of_Match'].value_counts().reset_index()
potm_full.columns = ['Player', 'Awards']

# Table 6: H2H (long format)
h2h_long = h2h.stack().reset_index()
h2h_long.columns = ['Winner', 'Loser', 'Wins']
h2h_long = h2h_long[h2h_long['Wins'] > 0]

# Write to Excel workbook (each table = one sheet)
excel_path = 'ipl_powerbi_export.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_clean.to_excel(writer,      sheet_name='Matches',        index=False)
    team_summary.to_excel(writer,  sheet_name='TeamSummary',    index=False)
    season_summary.to_excel(writer,sheet_name='SeasonSummary',  index=False)
    venue_summary.to_excel(writer, sheet_name='VenueSummary',   index=False)
    potm_full.to_excel(writer,     sheet_name='PlayerOfMatch',  index=False)
    h2h_long.to_excel(writer,      sheet_name='HeadToHead',     index=False)

print(f"✅ Power BI export ready: {excel_path}")
print("   Sheets: Matches | TeamSummary | SeasonSummary | VenueSummary | PlayerOfMatch | HeadToHead")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 22: MYSQL SCHEMA + INSERT SCRIPT
# ─────────────────────────────────────────────────────────────────────────────
sql_schema = """
-- ============================================================
-- IPL DATABASE SCHEMA  (MySQL 8.0+)
-- Run in MySQL Workbench or any MySQL client
-- ============================================================

CREATE DATABASE IF NOT EXISTS ipl_db CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;
USE ipl_db;

-- ── TEAMS ───────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS teams (
    team_id   INT AUTO_INCREMENT PRIMARY KEY,
    team_name VARCHAR(80) NOT NULL UNIQUE,
    short_name VARCHAR(5)
);

-- ── VENUES ──────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS venues (
    venue_id      INT AUTO_INCREMENT PRIMARY KEY,
    venue_name    VARCHAR(150) NOT NULL,
    city          VARCHAR(80),
    UNIQUE KEY uq_venue (venue_name)
);

-- ── SEASONS ─────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS seasons (
    season_id   INT AUTO_INCREMENT PRIMARY KEY,
    season_code VARCHAR(10) NOT NULL UNIQUE,
    season_year INT NOT NULL
);

-- ── MATCHES ─────────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS matches (
    match_id         INT PRIMARY KEY,
    season_id        INT,
    match_date       DATE,
    match_number     VARCHAR(20),
    team1_id         INT,
    team2_id         INT,
    venue_id         INT,
    toss_winner_id   INT,
    toss_decision    ENUM('bat','field'),
    winning_team_id  INT,
    won_by           ENUM('Runs','Wickets','SuperOver','NoResults'),
    margin           DECIMAL(6,2),
    super_over       TINYINT(1) DEFAULT 0,
    player_of_match  VARCHAR(60),
    FOREIGN KEY (season_id)       REFERENCES seasons(season_id),
    FOREIGN KEY (team1_id)        REFERENCES teams(team_id),
    FOREIGN KEY (team2_id)        REFERENCES teams(team_id),
    FOREIGN KEY (venue_id)        REFERENCES venues(venue_id),
    FOREIGN KEY (toss_winner_id)  REFERENCES teams(team_id),
    FOREIGN KEY (winning_team_id) REFERENCES teams(team_id)
);

-- ── PLAYER OF MATCH SUMMARY (denormalised view) ─────────────
CREATE OR REPLACE VIEW vw_potm_leaderboard AS
SELECT player_of_match AS player, COUNT(*) AS awards
FROM matches
WHERE player_of_match IS NOT NULL
GROUP BY player_of_match
ORDER BY awards DESC;

-- ── TEAM WIN SUMMARY ────────────────────────────────────────
CREATE OR REPLACE VIEW vw_team_wins AS
SELECT t.team_name,
       COUNT(m.match_id)         AS total_wins,
       COUNT(m.match_id) * 100.0 /
           (SELECT COUNT(*) FROM matches m2
            WHERE m2.team1_id = t.team_id OR m2.team2_id = t.team_id) AS win_rate_pct
FROM teams t
JOIN matches m ON m.winning_team_id = t.team_id
GROUP BY t.team_id, t.team_name
ORDER BY total_wins DESC;

-- ── SEASON SUMMARY ──────────────────────────────────────────
CREATE OR REPLACE VIEW vw_season_summary AS
SELECT s.season_year,
       COUNT(m.match_id)              AS total_matches,
       AVG(m.margin)                  AS avg_margin,
       SUM(m.super_over)              AS super_overs,
       SUM(CASE WHEN m.toss_winner_id = m.winning_team_id THEN 1 ELSE 0 END) * 100.0
           / COUNT(m.match_id)        AS toss_win_pct
FROM seasons s
JOIN matches m USING (season_id)
GROUP BY s.season_year
ORDER BY s.season_year;
"""

with open('ipl_schema.sql', 'w') as f:
    f.write(sql_schema)
print("✅ MySQL schema saved: ipl_schema.sql")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 23: MYSQL PYTHON INSERT (via SQLAlchemy or mysql-connector)
# ─────────────────────────────────────────────────────────────────────────────
mysql_insert_code = '''
# ── Replace with your MySQL credentials ──────────────────────
# pip install sqlalchemy pymysql
from sqlalchemy import create_engine

DB_USER = "root"
DB_PASS = "your_password"
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "ipl_db"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    echo=False
)

# ─────────────────────────────────────────────────
# Build team & venue dimension tables
# ─────────────────────────────────────────────────
import pandas as pd, numpy as np

df = pd.read_csv('ipl-matches.csv')
df['Date'] = pd.to_datetime(df['Date'])

def season_year(s):
    s = str(s)
    return int(s.split('/')[0]) if '/' in s else int(s)

df['SeasonYear'] = df['Season'].apply(season_year)
df['VenueShort'] = df['Venue'].str.split(',').str[0].str.strip()
df['Margin'] = pd.to_numeric(df['Margin'], errors='coerce')
df['IsSuperOver'] = df['SuperOver'].str.strip().str.upper().eq('Y').astype(int)

all_teams = pd.concat([df['Team1'], df['Team2'], df['TossWinner'],
                        df['WinningTeam'].dropna()]).unique()
teams_df  = pd.DataFrame({'team_name': sorted(all_teams)})
teams_df.to_sql('teams', engine, if_exists='append', index=False)
print(f"✅ Inserted {len(teams_df)} teams")

# Load back with IDs
teams_lookup = pd.read_sql("SELECT team_id, team_name FROM teams", engine)
team_map = dict(zip(teams_lookup['team_name'], teams_lookup['team_id']))

venues_df = (df[['Venue','City']].drop_duplicates()
               .rename(columns={'Venue':'venue_name','City':'city'}))
venues_df.to_sql('venues', engine, if_exists='append', index=False)
print(f"✅ Inserted {len(venues_df)} venues")
venue_map = pd.read_sql("SELECT venue_id, venue_name FROM venues", engine)
venue_map = dict(zip(venue_map['venue_name'], venue_map['venue_id']))

seasons_df = (df[['Season','SeasonYear']].drop_duplicates()
                .rename(columns={'Season':'season_code','SeasonYear':'season_year'}))
seasons_df.to_sql('seasons', engine, if_exists='append', index=False)
season_map_df = pd.read_sql("SELECT season_id, season_code FROM seasons", engine)
season_map = dict(zip(season_map_df['season_code'], season_map_df['season_id']))

# Build matches fact table
matches_insert = df[[\'ID\',\'Season\',\'Date\',\'MatchNumber\',\'Team1\',\'Team2\',
                       \'Venue\',\'TossWinner\',\'TossDecision\',\'WinningTeam\',
                       \'WonBy\',\'Margin\',\'IsSuperOver\',\'Player_of_Match\']].copy()

matches_insert[\'match_id\']        = matches_insert[\'ID\']
matches_insert[\'season_id\']       = matches_insert[\'Season\'].map(season_map)
matches_insert[\'match_date\']      = matches_insert[\'Date\']
matches_insert[\'match_number\']    = matches_insert[\'MatchNumber\']
matches_insert[\'team1_id\']        = matches_insert[\'Team1\'].map(team_map)
matches_insert[\'team2_id\']        = matches_insert[\'Team2\'].map(team_map)
matches_insert[\'venue_id\']        = matches_insert[\'Venue\'].map(venue_map)
matches_insert[\'toss_winner_id\']  = matches_insert[\'TossWinner\'].map(team_map)
matches_insert[\'toss_decision\']   = matches_insert[\'TossDecision\']
matches_insert[\'winning_team_id\'] = matches_insert[\'WinningTeam\'].map(team_map)
matches_insert[\'won_by\']          = matches_insert[\'WonBy\']
matches_insert[\'margin\']          = matches_insert[\'Margin\']
matches_insert[\'super_over\']      = matches_insert[\'IsSuperOver\']
matches_insert[\'player_of_match\'] = matches_insert[\'Player_of_Match\']

cols = [\'match_id\',\'season_id\',\'match_date\',\'match_number\',\'team1_id\',
         \'team2_id\',\'venue_id\',\'toss_winner_id\',\'toss_decision\',
         \'winning_team_id\',\'won_by\',\'margin\',\'super_over\',\'player_of_match\']
matches_insert[cols].to_sql(\'matches\', engine, if_exists=\'append\', index=False)
print(f"✅ Inserted {len(matches_insert)} match rows into MySQL")
'''

with open('ipl_mysql_insert.py', 'w') as f:
    f.write(mysql_insert_code)
print("✅ MySQL insert script saved: ipl_mysql_insert.py")
print("   Edit DB_USER / DB_PASS and run it after creating the schema.")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 24: KEY INSIGHTS SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
most_wins_team  = team_wins.iloc[0]['Team']
most_wins_count = team_wins.iloc[0]['Wins']
best_potm       = potm.iloc[0]['Player']
best_potm_n     = potm.iloc[0]['Awards']
busiest_venue   = venue_matches.index[0]
busiest_count   = venue_matches.values[0]
toss_pct        = df['TossWinMatchWin'].mean() * 100
runs_pct        = won_by_counts.get('Runs', 0) / len(df) * 100
wkts_pct        = won_by_counts.get('Wickets', 0) / len(df) * 100

print("""
╔══════════════════════════════════════════════════════════╗
║           🏆  IPL KEY INSIGHTS SUMMARY                  ║
╠══════════════════════════════════════════════════════════╣""")
print(f"║  Most Successful Team : {most_wins_team:<30} ({most_wins_count} wins) ║")
print(f"║  Best POTM Player     : {best_potm:<30} ({best_potm_n} awards)  ║")
print(f"║  Busiest Venue        : {busiest_venue:<30} ({busiest_count} matches) ║")
print(f"║  Toss Winner Match %  : {toss_pct:.1f}%                                 ║")
print(f"║  Win by Runs          : {runs_pct:.1f}%                                 ║")
print(f"║  Win by Wickets       : {wkts_pct:.1f}%                                 ║")
print(f"║  Total Super Overs    : {df['IsSuperOver'].sum():<38}  ║")
print("""╚══════════════════════════════════════════════════════════╝

📁 FILES GENERATED:
   01-11  → PNG charts (ready to embed in reports)
   12     → 12_city_matches.html  (interactive Plotly)
   13     → 13_ipl_dashboard.html (full interactive dashboard)
   14     → ipl_powerbi_export.xlsx  (6 sheets → connect in Power BI)
   15     → ipl_schema.sql           (MySQL schema + views)
   16     → ipl_mysql_insert.py      (Python → MySQL loader)

🔗 POWER BI STEPS:
   1. Open Power BI Desktop
   2. Get Data → Excel → ipl_powerbi_export.xlsx
   3. Load all 6 sheets
   4. Create relationships:
        Matches[WinningTeam] → TeamSummary[Team]
        Matches[SeasonYear]  → SeasonSummary[SeasonYear]
        Matches[VenueShort]  → VenueSummary[VenueShort]
   5. Build visuals matching the dashboard screenshot provided.
""")

✅ Libraries imported successfully!


FileNotFoundError: [Errno 2] No such file or directory: 'ipl-matches.csv'